# Barranquilla MOPAC: solids (Tv) on an A100

Validates the GPU support for materials on Colab (Runtime > Change runtime type > **A100 GPU**):

1. **check**: Si, quartz, rutile and urea cells of 144-192 atoms, 1SCF GRADIENTS with the device hcore, Fock and gradient compared to the CPU loops of the same run (`MOPAC_GPU_HCORE_CHECK`, `MOPAC_GPU_FOCK_CHECK`, `MOPAC_GPU_GRAD_CHECK`)
2. **medium**: quartz and rutile with 576 atoms and Si with 512 atoms, GPU time and heat compared to the Linux CPU references (4 vCPU: 240, 809, 1224 s)
3. **force**: diamond FORCE (64 atoms), GPU vs CPU (same binary, `MOPAC_NOGPU=1`)
4. **drc**: diamond DRC at 300 K, 40 steps, GPU vs CPU
5. stage profile of the medium cells (`MOPAC_GPU_PROFILE=1`)
6. optional: MOZYME (chignolin, crambin) against stored CPU references and the ctest suite

The source is cloned from GitHub (`juvenalyosa/mopac_gpu`, branch `main`). Everything is written to `/content/results` and downloaded as a zip in the last cell.

## 0. Options

In [ ]:
BRANCH = 'main'
RUN_CHECK = True        # 1: device hcore / Fock / gradient vs CPU (~5 min)
MEDIUM = 'quartz,silicon,rutile'   # 2: medium cells ('' to skip), ~2-6 min each
RUN_FORCE = True        # 3: diamond FORCE GPU and CPU (the CPU run takes ~10 min)
RUN_DRC = True          # 4: diamond DRC GPU and CPU (~4 min)
RUN_PROFILE = True      # 5: stage profile of the medium cells
RUN_PDIAG = True        # 4b: medium cells again with MOPAC_PDIAG=0 (full diagonalization every iteration)
RUN_MOZYME = False      # 6a: MOZYME chignolin / crambin vs stored CPU references (~15 min)
RUN_CTEST = False       # 6b: the MOPAC test suite with the GPU build (~20 min)
DPAIR_PROFILE = False   # phases of the d-pair kernels (hcore, gradient) in the medium cells; slows them a little


## 1. GPU

In [ ]:
import os, subprocess, time, re, json, shutil
from pathlib import Path

def run(cmd, log=None, check=True, env=None):
    """Run a shell command, stream its output, keep it in log."""
    print('$', cmd, flush=True)
    full = dict(os.environ); full.update(env or {})
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=full)
    fh = open(log, 'a') if log else None
    for line in p.stdout:
        print(line, end='', flush=True)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

RES = Path('/content/results'); RES.mkdir(parents=True, exist_ok=True)
run('nvidia-smi', RES / 'nvidia-smi.txt')
gpu = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
print('GPU:', gpu, '| CPUs:', os.cpu_count())
if 'A100' not in gpu:
    print('WARNING: not an A100 (Runtime > Change runtime type). The checks still run.')
CC = gpu.split(',')[-1].strip().replace('.', '')

## 2. Build (CUDA, Release)

In [ ]:
SRC, BUILD = Path('/content/mopac_gpu'), Path('/content/build')
run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gfortran ninja-build '
    'libblas-dev liblapack-dev > /dev/null')
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
shutil.rmtree(SRC, ignore_errors=True); shutil.rmtree(BUILD, ignore_errors=True)
run(f'git clone -q --depth 1 -b {BRANCH} https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
commit = subprocess.run(f'git -C {SRC} log -1 --format="%h %ad %s" --date=short', shell=True,
                        capture_output=True, text=True).stdout.strip()
print('commit:', commit)
(RES / 'commit.txt').write_text(commit + '\n' + gpu + '\n')
t0 = time.time()
run(f'cmake -S {SRC} -B {BUILD} -G Ninja -DGPU=ON -DTESTS=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_ARCHS={CC} '
    f'> {RES}/cmake.log 2>&1 || (tail -40 {RES}/cmake.log; exit 1)')
run(f'cmake --build {BUILD} --target mopac --parallel {os.cpu_count()} > {RES}/build.log 2>&1 '
    f'|| (grep -i -B2 -A6 error {RES}/build.log | head -80; exit 1)')
MOPAC = BUILD / 'mopac'
assert MOPAC.exists()
print(f'build OK in {time.time() - t0:.0f} s: {MOPAC}')
os.environ['OMP_NUM_THREADS'] = str(os.cpu_count())
if globals().get('DPAIR_PROFILE'):
    os.environ['MOPAC_GPU_DPAIR_PROFILE'] = '1'
CHECK = f'python3 {SRC}/tests/check_materials_gpu.py {MOPAC} --work-dir {RES}/materials'

## 3. Device hcore / Fock / gradient vs CPU (144-192 atoms)

In [ ]:
if RUN_CHECK:
    run(f"{CHECK} --medium '' --skip-force --skip-drc", RES / 'check.log', check=False)

## 4. Medium cells (512-576 atoms) vs CPU references

In [ ]:
if MEDIUM:
    run(f"{CHECK} --skip-check --medium {MEDIUM} --skip-force --skip-drc", RES / 'medium.log', check=False)

## 4b. Medium cells without the pseudo-diagonalization (MOPAC_PDIAG=0, reference)


In [ ]:
if RUN_PDIAG and MEDIUM:
    run(f"python3 {SRC}/tests/check_materials_gpu.py {MOPAC} --work-dir {RES}/materials_nopdiag --skip-check "
        f"--medium {MEDIUM} --skip-force --skip-drc", RES / 'medium_nopdiag.log', check=False, env={'MOPAC_PDIAG': '0'})

## 5. Diamond FORCE, GPU vs CPU

In [ ]:
if RUN_FORCE:
    run(f"{CHECK} --skip-check --medium '' --skip-drc", RES / 'force.log', check=False)

## 6. Diamond DRC (300 K), GPU vs CPU

In [ ]:
if RUN_DRC:
    run(f"{CHECK} --skip-check --medium '' --skip-force", RES / 'drc.log', check=False)

## 7. Stage profile of the medium cells

In [ ]:
def profile(path):
    out = Path(path).read_text(errors='ignore').splitlines()
    keep = [l.strip() for l in out if '[PROFILE]' in l and any(k in l for k in ('conv_', 'hcore_total', 'deriv', 'dcart'))]
    keep += [l.strip() for l in out if 'AFTER  HCORE' in l or 'AFTER  DERIV' in l or 'AFTER  ITER' in l]
    keep += [l.strip() for l in out if 'AFTER  FOCKS' in l][-1:] + [l.strip() for l in out if 'AFTER  FULL DIAG' in l][-1:]
    keep += [l.strip() for l in out if 'FINAL HEAT' in l or 'TOTAL JOB TIME' in l]
    keep.append(f"iterations {sum('TIME FOR ITERATION' in l for l in out)}")
    # TIMES: wall-clock seconds summed per stage over the whole run ('AFTER X' = stage X,
    # 'BEFORE X' = the untimed work since the previous mark)
    sums = {}
    for l in out:
        m = re.match(r'\s+(BEFORE|AFTER)\s+(.+?)\s+CPU INTERVAL:.*WALL-CLOCK INTERVAL:\s+([\d.]+)', l)
        if m:
            key = ('' if m.group(1) == 'AFTER' else 'gap before ') + m.group(2).strip()
            sums[key] = sums.get(key, 0.0) + float(m.group(3))
    for k, v in sorted(sums.items(), key=lambda kv: -kv[1])[:12]:
        keep.append(f'TIMES {k:28s} {v:8.2f} s')
    return keep

if RUN_PROFILE and MEDIUM:
    with open(RES / 'profile.txt', 'w') as fh:
        for tag in ('materials', 'materials_nopdiag'):
            for name in MEDIUM.split(','):
                f = RES / tag / 'medium' / name / f'{name}.out'
                if f.exists():
                    lines = [f'== {name} ({tag})'] + ['   ' + l for l in profile(f)]
                    so = f.with_suffix('.stdout')
                    if so.exists():
                        lines += ['   ' + l.strip() for l in so.read_text(errors='ignore').splitlines()
                                  if '[GPU dpair profile]' in l]
                    print('\n'.join(lines)); fh.write('\n'.join(lines) + '\n')

## 8. Optional: MOZYME references and ctest

In [ ]:
if RUN_MOZYME:
    run(f'python3 {SRC}/tests/check_mozyme_gpu_reference.py {MOPAC} --data {SRC}/tests/data/mozyme_gpu_reference '
        f'--work-dir {RES}/mozyme --workers 0 --concurrent 0 --skip-forcets', RES / 'mozyme.log', check=False)
if RUN_CTEST:
    run('pip -q install numpy')
    run(f'cmake -S {SRC} -B {BUILD} -DTESTS=ON > {RES}/ctest_configure.log 2>&1 && '
        f'cmake --build {BUILD} --parallel {os.cpu_count()} > {RES}/ctest_build.log 2>&1', check=False)
    # full log (with the output of the failed tests) in ctest_full.log, its end on screen
    run(f'cd {BUILD} && ctest -j {os.cpu_count()} --output-on-failure > {RES}/ctest_full.log 2>&1; '
        f'tail -40 {RES}/ctest_full.log', RES / 'ctest.log', check=False)
    # outputs of the failed tests (the comparison needs the whole .out)
    failed = re.findall(r'\d+ - (\S+) \(Failed\)', (RES / 'ctest_full.log').read_text(errors='ignore'))
    for name in failed:
        d = RES / 'ctest_failed' / name
        d.mkdir(parents=True, exist_ok=True)
        for f in list((BUILD / 'tests').glob(f'*{name}*')) + list((BUILD / 'tests').glob('*.dump')):
            if f.is_file():
                shutil.copy(f, d)
    print('failed tests:', failed)

## 9. Structure viewer

Every deck in `tests/data/materials` and every calculation run above: the cell (translation vectors) and a supercell, the DRC trajectory, and the FORCE normal modes (animated). Run it any time; *refresh list* picks up new results.

In [ ]:
# Structure viewer: every deck (tests/data/materials) and every calculation run above (results/materials):
# the cell with its translation vectors, a supercell, the DRC trajectory and the FORCE normal modes.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'py3Dmol'], check=False)
import re
from pathlib import Path
import numpy as np
import py3Dmol
import ipywidgets as w
from IPython.display import display, clear_output

SRC = Path(globals().get('SRC', '/content/mopac_gpu'))
RES = Path(globals().get('RES', '/content/results'))
STYLE = {'stick': {'radius': 0.12}, 'sphere': {'scale': 0.22}}


def read_mop(path):
    """Atoms (symbols, xyz) and translation vectors of a Cartesian MOPAC deck."""
    lines = Path(path).read_text(errors='ignore').splitlines()
    syms, xyz, tv = [], [], []
    for line in lines[3:]:
        t = line.split()
        if len(t) < 4:
            continue
        try:
            v = [float(t[i]) for i in ((1, 3, 5) if len(t) >= 7 else (1, 2, 3))]
        except ValueError:
            continue
        if t[0].lower() == 'tv':
            tv.append(v)
        else:
            syms.append(re.sub(r'[^A-Za-z].*', '', t[0]))
            xyz.append(v)
    return syms, np.array(xyz), np.array(tv)


def xyz_text(syms, xyz, title=''):
    return f'{len(syms)}\n{title}\n' + ''.join(f'{s} {x:.5f} {y:.5f} {z:.5f}\n' for s, (x, y, z) in zip(syms, xyz))


def supercell(syms, xyz, tv, n):
    if len(tv) == 0 or n == 1:
        return syms, xyz
    shifts = [i * tv[0] + (j * tv[1] if len(tv) > 1 else 0) + (k * tv[2] if len(tv) > 2 else 0)
              for i in range(n) for j in range(n if len(tv) > 1 else 1) for k in range(n if len(tv) > 2 else 1)]
    return syms * len(shifts), np.vstack([xyz + s for s in shifts])


def draw_cell(view, tv):
    if len(tv) == 0:
        return
    vec = [tv[i] if i < len(tv) else np.zeros(3) for i in range(3)]
    corners = {(i, j, k): i * vec[0] + j * vec[1] + k * vec[2] for i in (0, 1) for j in (0, 1) for k in (0, 1)}
    for c, p in corners.items():
        for d in range(3):
            if c[d] == 0 and np.linalg.norm(vec[d]) > 0:
                q = corners[tuple(1 if e == d else c[e] for e in range(3))]
                view.addCylinder({'start': dict(zip('xyz', map(float, p))), 'end': dict(zip('xyz', map(float, q))),
                                  'radius': 0.04, 'color': '#444444', 'fromCap': 1, 'toCap': 1})


def drc_frames(path):
    """Frames of a MOPAC .xyz trajectory (DRC / IRC)."""
    lines = Path(path).read_text(errors='ignore').splitlines()
    frames, i = [], 0
    while i < len(lines):
        try:
            n = int(lines[i].split()[0])
        except (ValueError, IndexError):
            break
        frames.append('\n'.join(lines[i:i + n + 2]) + '\n')
        i += n + 2
    return frames


def normal_modes(path):
    """(frequency, displacement 3N) of every root in NORMAL COORDINATE ANALYSIS of a FORCE output."""
    text = Path(path).read_text(errors='ignore')
    k = text.find('NORMAL COORDINATE ANALYSIS')
    if k < 0:
        return []
    end = text.find('MASS-WEIGHTED COORDINATE ANALYSIS', k)
    lines = text[k:end if end > 0 else None].splitlines()
    modes, cols, i = {}, [], 0
    while i < len(lines):
        t = lines[i].split()
        if lines[i].strip().startswith('Root No.'):
            cols = [int(x) for x in t[2:]]
            j = i + 1
            freq = None
            while j < len(lines) and freq is None:   # the frequency line: only numbers, as many as roots
                u = lines[j].split()
                if len(u) == len(cols) and all(re.fullmatch(r'-?\d+\.\d+', x) for x in u):
                    freq = [float(x) for x in u]
                j += 1
            for c, f in zip(cols, freq or []):
                modes[c] = [f, []]
            i = j
            continue
        if cols and len(t) == len(cols) + 1 and t[0].isdigit():
            for c, x in zip(cols, t[1:]):
                modes[c][1].append(float(x))
        i += 1
    return [(f, np.array(v)) for _, (f, v) in sorted(modes.items()) if v]


def catalogue():
    """label -> (deck .mop, output .out or None)"""
    cat = {}
    for p in sorted((SRC / 'tests/data/materials').glob('*/*.mop')):
        cat[f'deck  {p.parent.name}/{p.stem}'] = (p, None)
    for p in sorted((RES / 'materials').rglob('*.out')):
        mop = p.with_suffix('.mop')
        if mop.exists():
            cat[f'run   {p.relative_to(RES / "materials").parent}/{p.stem}'] = (mop, p)
    return cat


cat = catalogue()
pick = w.Dropdown(options=list(cat), description='structure', layout=w.Layout(width='520px'))
what = w.ToggleButtons(options=['cell', 'DRC trajectory', 'FORCE mode'], description='show')
ncell = w.IntSlider(value=1, min=1, max=3, description='supercell')
mode = w.Dropdown(options=[], description='mode', layout=w.Layout(width='320px'))
amp = w.FloatSlider(value=0.6, min=0.1, max=2.0, step=0.1, description='amplitude')
refresh = w.Button(description='refresh list')
out = w.Output()
modes_cache = {}


def update_modes(*_):
    mop, res = cat[pick.value]
    ms = normal_modes(res) if res else []
    modes_cache[pick.value] = ms
    mode.options = [(f'{i + 1}: {f:.1f} cm-1', i) for i, (f, _) in enumerate(ms)]


def show(*_):
    with out:
        clear_output(wait=True)
        mop, res = cat[pick.value]
        syms, xyz, tv = read_mop(mop)
        view = py3Dmol.view(width=900, height=600)
        info = f'{len(syms)} atoms, {len(tv)} translation vectors'
        if len(tv):
            info += ', |Tv| = ' + ', '.join(f'{np.linalg.norm(v):.3f}' for v in tv) + ' A'
        if what.value == 'DRC trajectory':
            traj = res.with_suffix('.xyz') if res else None
            frames = drc_frames(traj) if traj and traj.exists() else []
            if not frames:
                print('no DRC trajectory (.xyz) for this calculation'); return
            view.addModelsAsFrames(''.join(frames), 'xyz')
            view.setStyle(STYLE)
            view.animate({'loop': 'forward', 'interval': 80})
            info += f', {len(frames)} DRC frames'
        elif what.value == 'FORCE mode':
            ms = modes_cache.get(pick.value) or []
            if not ms or mode.value is None:
                print('no normal modes in this output (run a FORCE calculation)'); return
            f, d = ms[mode.value]
            d = d[:3 * len(syms)].reshape(-1, 3)
            frames = ''.join(xyz_text(syms, xyz + amp.value * np.sin(2 * np.pi * s / 20) * d / max(1e-9, np.abs(d).max()),
                                      f'{f:.1f}') for s in range(20))
            view.addModelsAsFrames(frames, 'xyz')
            view.setStyle(STYLE)
            view.animate({'loop': 'forward', 'interval': 60})
            info += f', mode {mode.value + 1}: {f:.1f} cm-1'
        else:
            s2, x2 = supercell(syms, xyz, tv, ncell.value)
            view.addModel(xyz_text(s2, x2), 'xyz')
            view.setStyle(STYLE)
        draw_cell(view, tv)
        if res:
            h = re.findall(r'FINAL HEAT OF FORMATION =\s+(-?\d+\.\d+)', res.read_text(errors='ignore'))
            if h:
                info += f', heat {h[-1]} kcal/mol'
        print(f'{pick.value}: {info}')
        view.zoomTo()
        view.show()


def do_refresh(*_):
    global cat
    cat = catalogue()
    pick.options = list(cat)


pick.observe(lambda c: (update_modes(), show()), names='value')
for ctl in (what, ncell, mode, amp):
    ctl.observe(show, names='value')
refresh.on_click(do_refresh)
display(w.VBox([w.HBox([pick, refresh]), what, w.HBox([ncell, mode, amp]), out]))
update_modes()
show()

## 10. Summary and download

In [ ]:
lines = [f'commit: {commit}', f'gpu: {gpu}']
for log in ('check', 'medium', 'medium_nopdiag', 'force', 'drc', 'mozyme', 'ctest'):
    p = RES / f'{log}.log'
    if p.exists():
        lines += [l.rstrip() for l in p.read_text(errors='ignore').splitlines()
                  if l.startswith(('PASS', 'FAIL', '[materials]', 'all checks', '[ctest]')) or 'tests passed' in l
                  or 'check(s) failed' in l]
summary = '\n'.join(lines)
print(summary)
(RES / 'SUMMARY.txt').write_text(summary + '\n')
shutil.make_archive('/content/materials_a100_results', 'zip', RES)
from google.colab import files
files.download('/content/materials_a100_results.zip')